# BudAI Native v3 Subagents & Conversational HITL Test
This notebook verifies the **Subagents Architecture** and **Conversational HITL** resumptions.

### Protocol Validation:
- `0:`: Text and **Reasoning** (Thinking) tokens.
- `8:`: Data Blocks (HTIL Interrupts, Refresh Signals).
- `9:`: Tool Call Triggers (UI Chart rendering).

In [25]:
import requests
import json
import uuid
import sys

In [26]:
BASE_URL = "http://localhost:8080"
EMAIL = "arnavpragya04@gmail.com"
PASSWORD = "Anee2507*"

## 1. Authentication

In [27]:
login_res = requests.post(f"{BASE_URL}/api/auth/login", json={
    "email": EMAIL,
    "password": PASSWORD
})
if login_res.status_code == 200:
    token = login_res.json()["token"]
    headers = {"Authorization": f"Bearer {token}"}
    print(f"✅ Logged in successfully.")
else:
    print(f"❌ Login failed: {login_res.text}")

✅ Logged in successfully.


## 2. v3 Multiplexed Stream Client

In [28]:
def test_stream(user_input, session_id=None, active_account_id="ALL"):
    if not session_id: session_id = str(uuid.uuid4())
    
    payload = {
        "session_id": session_id,
        "active_account_id": active_account_id,
        "messages": [{"role": "user", "content": user_input}]
    }
    
    response = requests.post(
        f"{BASE_URL}/api/chat/stream",
        json=payload,
        headers=headers,
        stream=True
    )
    
    print(f"--- RUNNING v3 STREAM [Session: {session_id}] ---\n")
    
    interrupt_payload = None
    
    for line in response.iter_lines():
        if line:
            decoded_line = line.decode('utf-8')
            
            if not decoded_line.startswith("data: "):
                continue
                
            data_str = decoded_line[6:]
            if data_str == "[DONE]":
                break
            
            try:
                parsed_data = json.loads(data_str)
            except json.JSONDecodeError:
                continue
            
            msg_type = parsed_data.get("type")
            
            if msg_type == "text-delta":
                delta = parsed_data.get("delta", "")
                print(delta, end="", flush=True)
            elif msg_type == "reasoning-delta":
                delta = parsed_data.get("delta", "")
                print("\033[90m" + delta + "\033[0m", end="", flush=True)
                    
            elif msg_type == "data-message_annotations":
                data_list = parsed_data.get("data", [])
                for item in data_list:
                    if item.get("type") == "htil_interrupt":
                        print("\n\n[HTIL INTERRUPT]: Clarification required.")
                        interrupt_payload = item.get("interrupts")
                    else:
                        print(f"\n[DATA]: {item}")
                        
            elif msg_type == "data-tool_calls":
                tool_data = parsed_data.get("data", {})
                print(f"\n[UI TOOL]: {tool_data.get('toolName')} -> {tool_data.get('args')}")
                
    print("\n\n--- STREAM FINISHED ---")
    return session_id, interrupt_payload

## 3. Execution

In [29]:
# TEST A: General Query (Verify tokens and reasoning)
# test_stream("Hello BudAI, tell me who you are and explain what reasoning tokens are.")

In [30]:
# TEST B: HTIL Trigger (Vague spending query to trigger ask_user)
sid, interrupt = test_stream(
    "Help me maintain positive cashflow in 2026 with respect to 2025 for my Wise account")

--- RUNNING v3 STREAM [Session: 2709bb1a-8744-46d5-8d6e-5f675bdf51c2] ---


[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}



[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}


I am analyzing your Wise account cash flow for 2025 to establish a baseline for maintaining positive cash flow in 2026. Let me first examine your historical transactions for that period.


[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'ty

In [33]:
# TEST B: HTIL Trigger (Vague spending query to trigger ask_user)
test_stream(
    "Can you explain in 1 paragraph please?", session_id=sid)

--- RUNNING v3 STREAM [Session: a1180367-2310-437e-bfc8-e554893f2c21] ---


[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}

[DATA]: {'type': 'thinking_context', 'status': 'Thinking'}


I'm here to help explain anything you'd like! Could you please specify the topic or concept you'd like me to explain in one paragraph? For example, you might want to understand investment strategies, budgeting tips, or anything related to personal finance. Let me know how I can assist!

--- STREAM FINISHED ---


('a1180367-2310-437e-bfc8-e554893f2c21', None)

## 4. Conversational HTIL Resume
Resume by providing the user's conversational reply.

In [ ]:
# if interrupt:
#     # Extract available options if provided in the value
#     # Note: in create_agent with ask_user, the value is often the tool arguments2
#     print(f"Interrupt details: {interrupt}")
    
#     user_reply = "Please look at my Wise account for last month."
#     print(f"Resuming {sid} with reply: {user_reply}")
    
#     resume_payload = {
#         "session_id": sid,
#         "messages": [{
#             "type": "htil_response",
#             "payload": {"user_message": user_reply}
#         }]
#     }
    
#     # Stream the resumption
#     response = requests.post(f"{BASE_URL}/api/chat/stream", json=resume_payload, headers=headers, stream=True)
#     for line in response.iter_lines():
#         if line:
#             decoded_line = line.decode('utf-8')
#             if not decoded_line.startswith("data: "):
#                 continue
#             data_str = decoded_line[6:]
#             if data_str == "[DONE]":
#                 break
#             try:
#                 parsed_data = json.loads(data_str)
#             except:
#                 continue
                
#             msg_type = parsed_data.get("type")
#             if msg_type == "text-delta":
#                 delta = parsed_data.get("delta", "")
#                 print(delta, end="", flush=True)
#             elif msg_type == "reasoning-delta":
#                 delta = parsed_data.get("delta", "")
#                 print("\033[90m" + delta + "\033[0m", end="", flush=True)
#             elif msg_type == "data-message_annotations":
#                 print(f"\n[DATA]: {parsed_data.get('data')}")
#             elif msg_type == "data-tool_calls":
#                 tool_data = parsed_data.get("data", {})
#                 print(f"\n[UI TOOL]: {tool_data.get('toolName')} -> {tool_data.get('args')}")
